In [1]:
import torch.nn as nn
import torch
import torch.nn.functional as F

In [2]:
# Example top k gating working

num_experts = 4
top_k = 2
n_embd = 32


mh_out = torch.randn(2, 4, n_embd)

topkgate_linear = nn.Linear(n_embd, num_experts) # (batches, tokens, embd_dim)

logits = topkgate_linear(mh_out)

top_k_logits, top_k_indices = logits.topk(top_k, dim = -1) # Get topk experts





In [3]:
print(mh_out.shape)
print(logits.shape)
print(top_k_logits.shape)
print(top_k_indices.shape)

torch.Size([2, 4, 32])
torch.Size([2, 4, 4])
torch.Size([2, 4, 2])
torch.Size([2, 4, 2])


In [4]:
print(top_k_indices)

tensor([[[2, 0],
         [1, 0],
         [3, 2],
         [1, 2]],

        [[0, 3],
         [2, 3],
         [2, 3],
         [2, 1]]])


In [5]:
print( top_k_logits)

tensor([[[0.5363, 0.2619],
         [0.6992, 0.0615],
         [0.8339, 0.7481],
         [0.5994, 0.4607]],

        [[0.3790, 0.3092],
         [0.2598, 0.1965],
         [0.4910, 0.2785],
         [0.4893, 0.4064]]], grad_fn=<TopkBackward0>)


In [6]:
zeros = torch.full_like(logits, float('-inf'))
sparse_logits = zeros.scatter(-1, top_k_indices, top_k_logits)
print(sparse_logits)

tensor([[[0.2619,   -inf, 0.5363,   -inf],
         [0.0615, 0.6992,   -inf,   -inf],
         [  -inf,   -inf, 0.7481, 0.8339],
         [  -inf, 0.5994, 0.4607,   -inf]],

        [[0.3790,   -inf,   -inf, 0.3092],
         [  -inf,   -inf, 0.2598, 0.1965],
         [  -inf,   -inf, 0.4910, 0.2785],
         [  -inf, 0.4064, 0.4893,   -inf]]], grad_fn=<ScatterBackward0>)


In [7]:
print(logits)

tensor([[[ 0.2619,  0.0796,  0.5363, -0.2338],
         [ 0.0615,  0.6992, -0.3126, -0.3514],
         [ 0.1495, -0.6590,  0.7481,  0.8339],
         [-0.4724,  0.5994,  0.4607, -0.4195]],

        [[ 0.3790, -0.5343,  0.0226,  0.3092],
         [-0.0633, -0.8978,  0.2598,  0.1965],
         [ 0.0816,  0.0942,  0.4910,  0.2785],
         [-1.1073,  0.4064,  0.4893,  0.3256]]], grad_fn=<ViewBackward0>)


In [8]:
gating_output = F.softmax(sparse_logits, dim=-1)
gating_output

tensor([[[0.4318, 0.0000, 0.5682, 0.0000],
         [0.3458, 0.6542, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.4786, 0.5214],
         [0.0000, 0.5346, 0.4654, 0.0000]],

        [[0.5174, 0.0000, 0.0000, 0.4826],
         [0.0000, 0.0000, 0.5158, 0.4842],
         [0.0000, 0.0000, 0.5529, 0.4471],
         [0.0000, 0.4793, 0.5207, 0.0000]]], grad_fn=<SoftmaxBackward0>)

In [9]:
import moe

In [10]:
top_k_gate = moe.TopkRouter(n_embd, num_experts, top_k)

gating_output, indices = top_k_gate(mh_out)
gating_output.shape, gating_output, indices

(torch.Size([2, 4, 4]),
 tensor([[[0.0000, 0.7751, 0.0000, 0.2249],
          [0.0000, 0.6093, 0.0000, 0.3907],
          [0.0000, 0.7972, 0.2028, 0.0000],
          [0.2441, 0.0000, 0.7559, 0.0000]],
 
         [[0.0000, 0.0000, 0.3620, 0.6380],
          [0.5749, 0.0000, 0.4251, 0.0000],
          [0.4919, 0.0000, 0.0000, 0.5081],
          [0.0000, 0.0000, 0.8014, 0.1986]]], grad_fn=<SoftmaxBackward0>),
 tensor([[[1, 3],
          [1, 3],
          [1, 2],
          [2, 0]],
 
         [[3, 2],
          [0, 2],
          [3, 0],
          [2, 3]]]))

In [11]:
noisy_top_k_gate = moe.NoisyTopkRouter(n_embd, num_experts, top_k)
gating_output, indices = noisy_top_k_gate(mh_out)
gating_output, indices = noisy_top_k_gate(mh_out)
gating_output.shape, gating_output, indices

(torch.Size([2, 4, 4]),
 tensor([[[0.0000, 0.5724, 0.4276, 0.0000],
          [0.0000, 0.0000, 0.4322, 0.5678],
          [0.3630, 0.6370, 0.0000, 0.0000],
          [0.0000, 0.5368, 0.4632, 0.0000]],
 
         [[0.8469, 0.0000, 0.1531, 0.0000],
          [0.6731, 0.0000, 0.0000, 0.3269],
          [0.7588, 0.0000, 0.2412, 0.0000],
          [0.6640, 0.0000, 0.3360, 0.0000]]], grad_fn=<SoftmaxBackward0>),
 tensor([[[1, 2],
          [3, 2],
          [1, 0],
          [1, 2]],
 
         [[0, 2],
          [0, 3],
          [0, 2],
          [0, 2]]]))

In [ ]:
drop_out = 0.1

sparse_moe = moe.SparseMoE(n_embd, num_experts, top_k)
final_out = sparse_moe
